# RAG: Baseline vs. Cross-Encoder Re-Ranking

**Document:** MCC Laws of Cricket, 2017 Code, 4th Edition (2026)

Two retrieval pipelines over the same chunked corpus:

- **Pipeline 1 (baseline):** query → bi-encoder top 10 (cosine similarity) → take top 3 → LLM → Answer 1
- **Pipeline 2 (re-ranking):** same top 10 → cross-encoder scores all 10 (query, chunk) pairs → re-sort, take top 3 → LLM (same prompt) → Answer 2

## 1. Setup

Install the packages once (inside the project's `.venv`). Uncomment and run if they are not installed yet.

In [1]:
# %pip install torch sentence-transformers pypdf numpy pandas openai python-dotenv

In [2]:
import os

# Hide the Hugging Face cache warning about symlinks on Windows (harmless).
# Must be set before the Hugging Face libraries are imported.
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download progress bars in the outputs

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from huggingface_hub.utils import logging as hf_logging

hf_logging.set_verbosity_error()   # hide the notice about not using a Hugging Face token (not needed)

CHUNKS_PATH = Path("output/chunks.jsonl")
ANSWERS_PATH = Path("output/answers.json")   # cache of LLM answers (free tier is rate-limited)

BI_ENCODER_NAME = "BAAI/bge-small-en-v1.5"   # 512-token input limit
# bge models expect this instruction in front of queries (not in front of the chunks)
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

TOP_K_RETRIEVE = 10   # candidates from the bi-encoder
TOP_K_CONTEXT = 3     # chunks passed to the LLM

# Bi-encoder: embeds the query and each chunk separately.
# Chunk vectors can be computed once in advance, so search is fast.
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)

# Cross-encoder: reads the query and a chunk together and outputs one relevance score.
# More accurate, but it has to run once per (query, chunk) pair at query time.
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)

print("Bi-encoder embedding size:", bi_encoder.get_embedding_dimension())

Bi-encoder embedding size: 384


### LLM

The LLM is called through **OpenRouter**, which has an OpenAI-compatible API, so the standard `openai`
client works with a different base URL. Models ending in `:free` cost no credits but are rate-limited,
so answers are cached in `output/answers.json` and only generated once per prompt.
`ask_llm` always calls the API (used for latency timing); `ask_llm_cached` reuses saved answers.

The API key is read from a `.env` file (`OPENROUTER_API_KEY=...`) that is kept out of the repo.
To rerun this notebook, create your own `.env` next to it with your key.

In [3]:
import openai
from dotenv import load_dotenv

load_dotenv()   # loads OPENROUTER_API_KEY from .env into the environment (the key is never printed)

client = OpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.environ["OPENROUTER_API_KEY"])
LLM_MODEL = "nvidia/nemotron-3.5-super-vl:free"   # free tier on OpenRouter; ":free" means no credits are charged


def ask_llm(prompt, system=None):
    """Send one prompt (plus an optional system message with rules) to the LLM and return its answer text."""
    messages = [{"role": "system", "content": system}] if system else []
    messages.append({"role": "user", "content": prompt})
    # The free tier is sometimes busy: it may return an empty response or a server / rate-limit error.
    # Wait and retry a few times before giving up.
    last_problem = None
    for attempt in range(4):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
                temperature=0,   # deterministic answers, so both pipelines are comparable
            )
            if resp.choices and resp.choices[0].message.content:
                return resp.choices[0].message.content
            last_problem = getattr(resp, "error", "empty response")
        except (openai.APIStatusError, openai.APIConnectionError) as e:
            last_problem = e
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"LLM returned no answer after 4 attempts: {last_problem}")


# Cache: prompt -> answer, saved in output/answers.json. Rerunning the notebook reuses saved answers
# instead of spending free-tier requests. (Latency measurements use ask_llm, which always calls the API.)
answer_cache = json.loads(ANSWERS_PATH.read_text(encoding="utf-8")) if ANSWERS_PATH.exists() else {}


def ask_llm_cached(prompt, system=None):
    """Like ask_llm, but returns the saved answer if this model has already answered this exact prompt."""
    key = f"{LLM_MODEL} || {system} || {prompt}" if system else f"{LLM_MODEL} || {prompt}"
    if key not in answer_cache:
        answer_cache[key] = ask_llm(prompt, system)
        ANSWERS_PATH.parent.mkdir(exist_ok=True)
        ANSWERS_PATH.write_text(json.dumps(answer_cache, indent=2, ensure_ascii=False), encoding="utf-8")
    return answer_cache[key]


# Quick check that the API responds (instant on reruns, because the answer is cached)
t0 = time.perf_counter()
print(ask_llm_cached("In one sentence: how many balls are in an over in cricket?"))
print(f"{time.perf_counter() - t0:.2f} s")

There are six balls in an over in cricket.
0.00 s


## 2. Load, clean and chunk the PDF

**What the PDF looks like** (90 pages):
- Every PDF page is a two-page spread of the printed booklet, and its first line is the pair of printed page numbers (e.g. `10 11`).
- Pages 1–4: cover, preface, history and contents. Pages 83–90: diagram-only appendices (C, D, E) and the index.
- Pages 5–82: the Preamble, Laws 1–42, Appendix A (definitions) and Appendix B (the bat).

**Decisions:**
- Keep only pages 5–82. The contents page and the index list every Law title, so they would match almost any query on keywords alone and clutter the top 10.
- One chunk per numbered clause (e.g. `21.17 Out from a No ball`). A clause is one self-contained rule, which keeps rules and their exceptions (often in another Law) in separate chunks.
- Every chunk starts with a header naming its Law and clause, because a sub-clause such as "They shall not…" doesn't say on its own what it is about.
- Long clauses are split at sub-clause boundaries into pieces of at most 200 tokens, each with the same header. `bge-small` reads up to 512 tokens, but small chunks are kept on purpose: they make it more likely that a rule and its exception land in different chunks, and they keep each chunk focused on one rule.

In [4]:
import re
import urllib.request

PDF_URL = "https://www.lords.org/getmedia/1d908298-5c44-468d-b6a7-e1414a1296e0/Laws-of-Cricket-2017-Code-4th-Edition-(2026)_3.pdf"
PDF_PATH = "LAWS OF CRICKET.pdf"

# The Laws are MCC copyright, so the PDF is not stored in the repo: download it if it isn't here
if not os.path.exists(PDF_PATH):
    request = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request) as resp, open(PDF_PATH, "wb") as f:
        f.write(resp.read())

pages = [p.extract_text() or "" for p in PdfReader(PDF_PATH).pages]
print(len(pages), "PDF pages")

90 PDF pages


### 2.1 Split the text into clauses

Patterns found by looking at the extracted text:
- Law headings: `LAW 21 NO BALL`; appendix headings: `APPENDIX A DEFINITIONS AND EXPLANATIONS`.
- Clause headings: number, then non-breaking spaces (sometimes ordinary spaces), then a title starting with a capital, e.g. `21.17  Out from a No ball`, `A.4  Positioning`.
- Sub-clauses: `21.17.1 …`, `A.6.2 …`, and once `42 .4.2.3.1` with a stray space.

In [5]:
FIRST_PAGE, LAST_PAGE = 5, 82   # 1-based PDF pages: Preamble ... end of Appendix B

SPREAD_RE = re.compile(r"^\d+ \d+\s*$")
LAW_RE    = re.compile(r"^LAW (\d+) (.+)$")
APPX_RE   = re.compile(r"^APPENDIX ([AB])\s+(.+)$")
CLAUSE_RE = re.compile(r"^([AB]\.\d{1,2}|\d{1,2}\.\d{1,2})(?:\xa0+ ?| {1,3})(?=[A-Z])(.+)$")
SUB_RE    = re.compile(r"^\s*(?:[AB]\.\d{1,2}(?:\.\d{1,2})+|\d{1,2} ?\.\d{1,2}\.\d{1,2}(?:\.\d{1,2})*)\s")


def clean(s):
    """Normalise spaces and fix small extraction glitches."""
    s = s.replace("\xa0", " ").replace("\u2019 s", "\u2019s")   # "Striker’ s" -> "Striker’s"
    s = re.sub(r"(\d) \.(\d)", r"\1.\2", s)                     # "42 .4" -> "42.4"
    return re.sub(r"[ \t]+", " ", s).strip()


def nice_title(s):
    """'NO BALL' -> 'No ball'"""
    return clean(s).capitalize().replace("(law", "(Law")


sections = []
law_id, law_title = "Preamble", "The Spirit of Cricket"
current = None


def new_section(clause, clause_title, page):
    sec = {"law_id": law_id, "law_title": law_title, "clause": clause,
           "clause_title": clause_title, "page": page, "lines": []}
    sections.append(sec)
    return sec


def is_title_continuation(raw):
    """Second line of a long clause title: short, no sub-clause number, no full stop."""
    line = clean(raw)
    return 0 < len(line) <= 45 and not SUB_RE.match(raw) and not line.endswith(".")


title_may_wrap = False
for page_no in range(FIRST_PAGE, LAST_PAGE + 1):
    lines = pages[page_no - 1].split("\n")
    if lines and SPREAD_RE.match(lines[0]):
        lines = lines[1:]                                   # drop the "10 11" page-number line
    for raw in lines:
        if m := LAW_RE.match(raw):
            law_id, law_title = f"Law {m.group(1)}", nice_title(m.group(2))
            current = new_section(m.group(1), "", page_no)  # any text before the first clause
        elif m := APPX_RE.match(raw):
            law_id, law_title = f"Appendix {m.group(1)}", nice_title(m.group(2))
            current = new_section(m.group(1), "", page_no)
        elif m := CLAUSE_RE.match(raw):
            current = new_section(m.group(1), clean(m.group(2)), page_no)
            title_may_wrap = raw != raw.rstrip()            # wrapped titles end in a space
            continue
        elif title_may_wrap and is_title_continuation(raw):
            current["clause_title"] += " " + clean(raw)     # e.g. 2.8 "... unreasonable" + "circumstances"
        else:
            if current is None:
                current = new_section("Preamble", "", page_no)
            current["lines"].append(raw)
        title_may_wrap = False

# Sanity check: the index at the back lists every numbered clause of Laws 1-42
index_clauses = {m.group(1) for t in pages[83:] for ln in t.split("\n")
                 if (m := re.match(r"^(\d{1,2}\.\d{1,2})\s", ln))}
missing = index_clauses - {s["clause"] for s in sections}
print(f"{len(sections)} sections | index lists {len(index_clauses)} clauses | missing: {sorted(missing) or 'none'}")

339 sections | index lists 273 clauses | missing: none


### 2.2 Build the chunks

Lines are joined back into paragraphs, one per sub-clause. Paragraphs are then packed into chunks of at most 200 tokens (counted with the bi-encoder's own tokenizer). A single paragraph that is too long on its own is split at sentence boundaries.

In [6]:
MAX_TOKENS = 200   # well under bge-small's 512-token limit; small chunks keep one rule per chunk


def n_tokens(text):
    return len(bi_encoder.tokenizer.encode(text, add_special_tokens=False, verbose=False))


def to_paragraphs(lines):
    """Join wrapped lines; start a new paragraph at each sub-clause number or bullet."""
    paras = []
    for raw in lines:
        line = clean(raw)
        if not line:
            continue
        if paras and not (SUB_RE.match(raw) or raw.startswith("   ")):
            paras[-1] += line if paras[-1].endswith("-") else " " + line   # keep "wicket-keeper" intact
        else:
            paras.append(line)
    return paras


def split_long(para, budget):
    """Split a paragraph that is too long on its own at sentence boundaries."""
    if n_tokens(para) <= budget:
        return [para]
    parts, group = [], ""
    for sent in re.split(r"(?<=[.;:])\s+(?=[A-Z0-9])", para):
        if group and n_tokens(group + " " + sent) > budget:
            parts.append(group)
            group = sent
        else:
            group = f"{group} {sent}".strip()
    return parts + [group]


chunks = []


def add_chunk(sec, header, paras):
    chunks.append({"chunk_id": len(chunks), "law": sec["law_id"], "clause": sec["clause"],
                   "page": sec["page"], "text": header + "\n" + "\n".join(paras)})


for sec in sections:
    header = f"{sec['law_id']} – {sec['law_title']}"
    if sec["clause_title"]:
        header += f" | {sec['clause']} {sec['clause_title']}"
    budget = MAX_TOKENS - n_tokens(header)
    pieces = [p for para in to_paragraphs(sec["lines"]) for p in split_long(para, budget)]
    group = []
    for piece in pieces:
        if group and n_tokens("\n".join(group + [piece])) > budget:
            add_chunk(sec, header, group)
            group = []
        group.append(piece)
    if group:                      # sections with a heading but no text are skipped
        add_chunk(sec, header, group)

chunks_df = pd.DataFrame(chunks)
chunks_df["tokens"] = chunks_df["text"].map(n_tokens)
print(f"{len(chunks_df)} chunks | tokens min {chunks_df.tokens.min()}, "
      f"median {int(chunks_df.tokens.median())}, max {chunks_df.tokens.max()} | over 512: {(chunks_df.tokens > 512).sum()}")

432 chunks | tokens min 15, median 120, max 200 | over 512: 0


In [7]:
# Example chunk: the exception list for being out from a No ball
print(chunks_df.loc[chunks_df.clause == "21.17", "text"].iloc[0])

Law 21 – No ball | 21.17 Out from a No ball
When No ball has been called, neither batter shall be out under any of the Laws except 34 (Hit the ball twice), 37 (Obstructing the field) or 38 (Run out).


In [8]:
chunks_df[["chunk_id", "law", "clause", "page", "tokens", "text"]].head(10)

,chunk_id,law,clause,page,tokens,text
0,0,Preamble,Preamble,5,163,Preamble – The Spirit of Cricket\nTHE PREAMBLE...
1,1,Preamble,Preamble,5,110,Preamble – The Spirit of Cricket\nThank the of...
2,2,Law 1,1.1,5,112,Law 1 – The players | 1.1 Number of players\nA...
3,3,Law 1,1.2,5,173,Law 1 – The players | 1.2 Nomination and repla...
4,4,Law 1,1.3,6,124,Law 1 – The players | 1.3 Captain\n1.3.1 If at...
5,5,Law 1,1.4,6,63,Law 1 – The players | 1.4 Responsibility of ca...
6,6,Law 2,2.1,6,78,Law 2 – The umpires | 2.1 Appointment and atte...
7,7,Law 2,2.2,6,73,Law 2 – The umpires | 2.2 Change of umpire\nAn...
8,8,Law 2,2.3,6,194,Law 2 – The umpires | 2.3 Consultation with ca...
9,9,Law 2,2.3,6,97,Law 2 – The umpires | 2.3 Consultation with ca...


### 2.3 Save the chunks

Saved to `output/` so they can be inspected outside the notebook:
- `chunks.csv`: one row per chunk; opens in Excel (UTF-8 with BOM so the dashes and apostrophes display correctly).
- `chunks.jsonl`: one JSON object per line, easy to reload with `pd.read_json(..., lines=True)`.

These files contain the MCC text, so like the PDF they are kept out of the GitHub repo.

In [9]:
CHUNKS_PATH.parent.mkdir(exist_ok=True)
cols = ["chunk_id", "law", "clause", "page", "tokens", "text"]
chunks_df[cols].to_csv("output/chunks.csv", index=False, encoding="utf-8-sig")
chunks_df[cols].to_json(CHUNKS_PATH, orient="records", lines=True, force_ascii=False)
print("Saved", len(chunks_df), "chunks to output/chunks.csv and output/chunks.jsonl")

Saved 432 chunks to output/chunks.csv and output/chunks.jsonl


## 3. Embed and index the chunks

Every chunk is turned into a 384-number vector by the bi-encoder **once**, before any query arrives.
This is possible because a bi-encoder embeds each chunk on its own, without seeing the query.
(A cross-encoder cannot do this: it needs the query and the chunk together.)

- Chunks are embedded without the bge query prefix; the prefix is only added to queries.
- Vectors are normalised to length 1, so the dot product of two vectors equals their cosine similarity.
- The "vector store" is a plain numpy matrix (432 × 384). With so few chunks, comparing a query against
  every chunk is exact and takes well under a millisecond, so FAISS or Chroma isn't needed.
- The index is saved to `output/index.npz` and reused on later runs, unless the chunks or the model have changed
  (checked with a fingerprint of both). The build time shown is from when it was actually built.

In [10]:
import hashlib

INDEX_PATH = Path("output/index.npz")
# Fingerprint of the model name + all chunk texts: if either changes, the saved index is out of date
index_key = hashlib.sha256((BI_ENCODER_NAME + "\n" + "\n\n".join(chunks_df["text"])).encode("utf-8")).hexdigest()

saved_key = None
if INDEX_PATH.exists():
    with np.load(INDEX_PATH) as saved:
        saved_key = str(saved["key"])
        if saved_key == index_key:
            chunk_embeddings = saved["embeddings"]
            index_time = float(saved["build_seconds"])

if saved_key == index_key:
    print("Loaded saved index (same model and chunks as last time)")
else:
    t0 = time.perf_counter()
    chunk_embeddings = bi_encoder.encode(
        chunks_df["text"].tolist(),
        batch_size=32,
        normalize_embeddings=True,   # unit length -> dot product = cosine similarity
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    index_time = time.perf_counter() - t0
    np.savez(INDEX_PATH, embeddings=chunk_embeddings, key=index_key, build_seconds=index_time)
    print("Built a new index and saved it")

norms = np.linalg.norm(chunk_embeddings, axis=1)
print(f"Index: {chunk_embeddings.shape[0]} chunks x {chunk_embeddings.shape[1]} dims, "
      f"build time {index_time:.1f} s (one-off cost)")
print(f"Vector lengths: {norms.min():.4f} to {norms.max():.4f}")

Loaded saved index (same model and chunks as last time)
Index: 432 chunks x 384 dims, build time 26.4 s (one-off cost)
Vector lengths: 1.0000 to 1.0000


Quick check that the vectors make sense: the chunks most similar to the No ball exception (21.17)
should be other chunks about No balls or ways of getting out.

In [11]:
probe = chunks_df.index[chunks_df.clause == "21.17"][0]
sims = chunk_embeddings @ chunk_embeddings[probe]      # cosine similarity to every chunk
for i in np.argsort(-sims)[1:6]:                        # skip position 0: the chunk itself
    print(f"{sims[i]:.3f}  {chunks_df.text[i].splitlines()[0]}")

0.890  Law 22 – Wide ball | 22.9 Out from a Wide
0.865  Law 38 – Run out | 38.1 Out Run out
0.864  Law 38 – Run out | 38.2 Batter not out Run out
0.859  Law 21 – No ball | 21.14 Penalty for a No ball
0.859  Law 37 – Obstructing the field | 37.1 Out Obstructing the field


## 4. Pipeline functions

| Function | What it does |
|---|---|
| `retrieve_top10(query)` | Bi-encoder: embed the query (with the bge prefix), cosine similarity against every chunk, return the top 10 |
| `rerank(query, candidates)` | Cross-encoder: score each (query, chunk) pair, re-sort by that score |
| `generate(query, chunks)` | LLM: answer from the given chunks only, using one fixed prompt for both pipelines |
| `pipeline_baseline(query)` | top 10 → first 3 → LLM → **Answer 1** |
| `pipeline_rerank(query)` | same top 10 → cross-encoder → top 3 → LLM → **Answer 2** |

Every stage is timed separately (query embedding, vector search, cross-encoding, LLM), for the latency table later.

### 4.1 Retrieval and re-ranking

In [12]:
def retrieve_top10(query, k=TOP_K_RETRIEVE):
    """Pipeline 1, stage 1: bi-encoder search over the whole index."""
    t0 = time.perf_counter()
    # bge expects the instruction prefix on queries only
    q_vec = bi_encoder.encode(QUERY_PREFIX + query, normalize_embeddings=True, convert_to_numpy=True)
    t1 = time.perf_counter()
    scores = chunk_embeddings @ q_vec          # cosine similarity with all 432 chunks at once
    top = np.argsort(-scores)[:k]
    t2 = time.perf_counter()

    candidates = chunks_df.loc[top, ["chunk_id", "law", "clause", "text"]].reset_index(drop=True)
    candidates["bi_rank"] = np.arange(1, k + 1)
    candidates["bi_score"] = scores[top]
    return candidates, {"embed_query": t1 - t0, "vector_search": t2 - t1}


def rerank(query, candidates):
    """Pipeline 2, stage 2: score every (query, chunk) pair with the cross-encoder and re-sort."""
    t0 = time.perf_counter()
    # No bge prefix here: the cross-encoder reads the plain question together with each chunk
    ce_scores = cross_encoder.predict([(query, text) for text in candidates["text"]], show_progress_bar=False)
    t1 = time.perf_counter()

    ranked = candidates.copy()
    ranked["ce_score"] = ce_scores
    ranked = ranked.sort_values("ce_score", ascending=False).reset_index(drop=True)
    ranked["ce_rank"] = np.arange(1, len(ranked) + 1)
    return ranked, {"cross_encode": t1 - t0}

### 4.2 Generation

The same prompt is used for both pipelines, so any difference between Answer 1 and Answer 2
comes only from which chunks were retrieved.

The LLM must not fill gaps from its own cricket knowledge, otherwise a retrieval failure would be hidden
by a correct answer from memory. So the prompt:
- puts the rules in a **system message**, separate from the context and question (models follow system
  instructions more strictly);
- requires every statement to **quote the exact supporting words** from a numbered passage, which is hard
  to do for a fact the passages don't contain;
- makes **refusing the expected behaviour**: if no passage states the answer, reply exactly
  "The context does not say.", even if the model knows the answer from elsewhere.

Two further rules came from the failure analysis in section 6:
- The question may use slang the Laws never use ("beamer", "grubber"). The model may use its general understanding of
  what such a word *describes* to find the matching passage, but must say which passage it matched and why, and the
  rule itself must still come from the passage. This lets in a small, controlled amount of outside knowledge (word
  meanings, not rules).
- If the passages contain only exceptions ("not out" cases) and not the main rule, the model must say so, because an
  answer listing only exceptions can mislead even when every sentence in it is true.

A model's built-in knowledge can't be removed, only constrained like this. Section 5.1 tests whether it still refuses
when the answer is not in the passages.

In [13]:
SYSTEM_PROMPT = """You answer questions about the Laws of Cricket using ONLY the numbered context passages in the user's message.

Rules:
1. Every rule, number and consequence in your answer must come from a passage. After each statement, give the passage number and quote the exact supporting words, for example: [2] "within 3 minutes of the dismissal".
2. Do not use knowledge of the Laws of Cricket from outside the passages, even if you are sure it is correct.
3. The question may use everyday cricket words that the passages don't use (for example "beamer", "grubber", "Mankad"). You may use your general understanding of what such a word describes in order to find the passage that covers it. If you do, first say in one sentence which passage you matched it to and why. The rule itself must still come from the passage.
4. If no passage covers the question, reply exactly: The context does not say.
5. If the passages answer only part of the question, answer that part and say which part the context does not cover. If the passages contain only exceptions (for example "not out" cases) but not the main rule, say clearly that the main rule is not in the passages.
6. Keep the answer short."""

PROMPT_TEMPLATE = """Context passages:
{context}

Question: {question}"""


def build_prompt(query, chunks):
    context = "\n\n".join(f"[{i}] {text}" for i, text in enumerate(chunks["text"], start=1))
    return PROMPT_TEMPLATE.format(context=context, question=query)


def generate(query, chunks, use_cache=True):
    """LLM stage. use_cache=False forces a real API call (needed when timing)."""
    prompt = build_prompt(query, chunks)
    t0 = time.perf_counter()
    answer = ask_llm_cached(prompt, SYSTEM_PROMPT) if use_cache else ask_llm(prompt, SYSTEM_PROMPT)
    return answer, {"llm": time.perf_counter() - t0}

### 4.3 The two pipelines

`pipeline_rerank` can be given the candidates from `pipeline_baseline`, so both pipelines work on exactly
the same top 10 (as the brief requires) and the bi-encoder doesn't run twice.

In [14]:
def pipeline_baseline(query, use_cache=True):
    """Pipeline 1: bi-encoder top 10 -> first 3 -> LLM -> Answer 1."""
    candidates, timings = retrieve_top10(query)
    context = candidates.head(TOP_K_CONTEXT)
    answer, t_llm = generate(query, context, use_cache)
    return {"answer": answer, "context": context, "candidates": candidates, "timings": {**timings, **t_llm}}


def pipeline_rerank(query, candidates=None, use_cache=True):
    """Pipeline 2: same top 10 -> cross-encoder -> top 3 -> LLM -> Answer 2."""
    timings = {}
    if candidates is None:
        candidates, timings = retrieve_top10(query)
    ranked, t_ce = rerank(query, candidates)
    context = ranked.head(TOP_K_CONTEXT)
    answer, t_llm = generate(query, context, use_cache)
    return {"answer": answer, "context": context, "ranked": ranked, "timings": {**timings, **t_ce, **t_llm}}

### 4.4 Rank comparison table

For one query: all 10 candidates in bi-encoder order, with both ranks and scores, the rank shift,
and which chunks each pipeline passes to the LLM. Bi-encoder scores are cosine similarities;
cross-encoder scores are on a different scale, so only the order within each column is comparable.

`side_by_side(query)` shows the same 10 chunks as two ranked lists next to each other: the bi-encoder's order on the left and the cross-encoder's order on the right, with each chunk's original bi-encoder rank, so the re-ordering can be read at a glance.

In [15]:
def rank_table(query):
    candidates, _ = retrieve_top10(query)
    ranked, _ = rerank(query, candidates)
    t = ranked.sort_values("bi_rank").reset_index(drop=True)
    t["chunk"] = t["text"].str.split("\n").str[0].str.split(" | ", regex=False).str[-1]   # e.g. "21.17 Out from a No ball"
    t["shift"] = [f"#{b} → #{c}" for b, c in zip(t["bi_rank"], t["ce_rank"])]
    t["used_by"] = [
        "both" if b <= TOP_K_CONTEXT and c <= TOP_K_CONTEXT
        else "baseline" if b <= TOP_K_CONTEXT
        else "re-rank" if c <= TOP_K_CONTEXT
        else ""
        for b, c in zip(t["bi_rank"], t["ce_rank"])
    ]
    cols = ["chunk_id", "law", "chunk", "bi_rank", "bi_score", "ce_rank", "ce_score", "shift", "used_by"]
    return t[cols].round({"bi_score": 3, "ce_score": 2})


def side_by_side(query):
    """The same 10 chunks in two orders: bi-encoder ranking (left) and cross-encoder ranking (right)."""
    candidates, _ = retrieve_top10(query)
    ranked, _ = rerank(query, candidates)
    title = lambda text: text.split("\n")[0].split(" | ")[-1]
    bi = candidates.sort_values("bi_rank")
    return pd.DataFrame({
        "rank": np.arange(1, len(bi) + 1),
        "bi-encoder order": [title(t) for t in bi["text"]],
        "bi score": bi["bi_score"].round(3).values,
        "cross-encoder order": [title(t) for t in ranked["text"]],
        "ce score": ranked["ce_score"].round(2).values,
        "was bi #": ranked["bi_rank"].values,
    }).set_index("rank")


# Demo (retrieval only, no LLM call)
demo_q = "How many balls are there in an over?"
display(rank_table(demo_q))
side_by_side(demo_q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,132,Law 17,17.1 Number of balls,1,0.811,1,5.75,#1 → #1,both
1,204,Law 21,21.16 No ball not to count,2,0.746,8,1.06,#2 → #8,baseline
2,213,Law 22,22.8 Wide not to count,3,0.742,6,2.63,#3 → #6,baseline
3,135,Law 17,17.3 Validity of balls,4,0.723,3,3.51,#4 → #3,re-rank
4,134,Law 17,17.3 Validity of balls,5,0.721,4,3.47,#5 → #4,
5,186,Law 20,20.7 Dead ball; ball counting as one of over,6,0.719,2,4.11,#6 → #2,re-rank
6,136,Law 17,17.4 Call of Over,7,0.712,7,1.91,#7 → #7,
7,137,Law 17,17.5 Umpire miscounting,8,0.697,5,3.16,#8 → #5,
8,34,Law 4,4.4 New ball in match of more than one day’s d...,9,0.696,10,-1.64,#9 → #10,
9,185,Law 20,20.7 Dead ball; ball counting as one of over,10,0.692,9,0.35,#10 → #9,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,17.1 Number of balls,0.811,17.1 Number of balls,5.75,1
2,21.16 No ball not to count,0.746,20.7 Dead ball; ball counting as one of over,4.11,6
3,22.8 Wide not to count,0.742,17.3 Validity of balls,3.51,4
4,17.3 Validity of balls,0.723,17.3 Validity of balls,3.47,5
5,17.3 Validity of balls,0.721,17.5 Umpire miscounting,3.16,8
6,20.7 Dead ball; ball counting as one of over,0.719,22.8 Wide not to count,2.63,3
7,17.4 Call of Over,0.712,17.4 Call of Over,1.91,7
8,17.5 Umpire miscounting,0.697,21.16 No ball not to count,1.06,2
9,4.4 New ball in match of more than one day’s d...,0.696,20.7 Dead ball; ball counting as one of over,0.35,10


## 5. Sanity check

Easy questions whose answer sits in one obvious clause. Both pipelines should find it and give the
same correct answer. This checks that retrieval, re-ranking and generation work end to end before
designing the failure queries.

In [16]:
def run_both(query, use_cache=True):
    """Run both pipelines on the same top 10 and print the chunks used and both answers."""
    p1 = pipeline_baseline(query, use_cache)
    p2 = pipeline_rerank(query, candidates=p1["candidates"], use_cache=use_cache)
    header = lambda text: text.split("\n")[0]
    print("QUESTION:", query)
    print("\nPipeline 1 (baseline) top 3:")
    for r in p1["context"].itertuples():
        print(f"  bi #{r.bi_rank}  {header(r.text)}")
    print("Pipeline 2 (re-ranked) top 3:")
    for r in p2["context"].itertuples():
        print(f"  ce #{r.ce_rank} (was bi #{r.bi_rank})  {header(r.text)}")
    print("\nANSWER 1:", p1["answer"])
    print("\nANSWER 2:", p2["answer"])
    return p1, p2


SANITY_QUERIES = [
    "How many balls are there in an over?",
    "How many players are there in each side?",
    "How much time does an incoming batter have before they can be given out Timed out?",
]

In [17]:
p1, p2 = run_both(SANITY_QUERIES[0])

QUESTION: How many balls are there in an over?

Pipeline 1 (baseline) top 3:
  bi #1  Law 17 – The over | 17.1 Number of balls
  bi #2  Law 21 – No ball | 21.16 No ball not to count
  bi #3  Law 22 – Wide ball | 22.8 Wide not to count
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #1)  Law 17 – The over | 17.1 Number of balls
  ce #2 (was bi #6)  Law 20 – Dead ball | 20.7 Dead ball; ball counting as one of over
  ce #3 (was bi #4)  Law 17 – The over | 17.3 Validity of balls

ANSWER 1: An over consists of 6 valid balls. [1] "overs of 6 valid balls". A No ball does not count as one of the over. [2] "A No ball shall not count as one of the over." A Wide does not count as one of the over. [3] "A Wide shall not count as one of the over."

ANSWER 2: [1] "overs of 6 valid balls"


In [18]:
p1, p2 = run_both(SANITY_QUERIES[1])

QUESTION: How many players are there in each side?

Pipeline 1 (baseline) top 3:
  bi #1  Law 1 – The players | 1.1 Number of players
  bi #2  Appendix A – Definitions and explanations | A.7 Fielders
  bi #3  Appendix A – Definitions and explanations | A.6 Batters
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #1)  Law 1 – The players | 1.1 Number of players
  ce #2 (was bi #2)  Appendix A – Definitions and explanations | A.7 Fielders
  ce #3 (was bi #3)  Appendix A – Definitions and explanations | A.6 Batters

ANSWER 1: Each side has eleven players. [1] "A match is played between two sides, each of eleven players, one of whom shall be captain." By agreement, sides may have fewer or more than eleven, but not more than eleven may field at any time. [1] "By agreement a match may be played between sides of fewer than, or more than, eleven players, but not more than eleven players may field at any time."

ANSWER 2: Each side has eleven players. [1] "A match is played between two sides, each

In [19]:
p1, p2 = run_both(SANITY_QUERIES[2])

QUESTION: How much time does an incoming batter have before they can be given out Timed out?

Pipeline 1 (baseline) top 3:
  bi #1  Law 40 – Timed out | 40.1 Out Timed out
  bi #2  Law 41 – Unfair play | 41.10 Batter wasting time
  bi #3  Law 24 – Fielder’s absence; substitutes | 24.2 Fielder absent or leaving the field of play
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #1)  Law 40 – Timed out | 40.1 Out Timed out
  ce #2 (was bi #2)  Law 41 – Unfair play | 41.10 Batter wasting time
  ce #3 (was bi #10)  Law 25 – Batter’s innings; runners | 25.3 Restriction on batter commencing an innings

ANSWER 1: An incoming batter must be ready to receive the ball within 3 minutes of the dismissal or retirement; if not, they are out Timed out. [1] "within 3 minutes of the dismissal or retirement" and [1] "If this requirement is not met, the incoming batter will be out, Timed out."

ANSWER 2: An incoming batter has 3 minutes. [1] "within 3 minutes of the dismissal or retirement" and "If this requ

### 5.1 Does the LLM answer from the passages or from memory?

Two quick tests of the prompt:
- **Wrong context:** an easy question ("How many balls are there in an over?") with three unrelated passages
  (bat size, the ball, rolling the pitch). A model that sticks to the context must reply "The context does not say";
  answering "6" would mean it used its own knowledge.
- **Changed fact:** the real Timed out passage (40.1) with "3 minutes" changed to "5 minutes". A model that follows the
  context answers 5 minutes; answering 3 would mean it trusts its memory over the passage.

In [20]:
wrong_context = chunks_df[chunks_df["clause"].isin(["5.7", "4.1", "9.1"])].drop_duplicates("clause")
print("Passages given:", ", ".join(wrong_context["clause"]))
print("Answer:", generate("How many balls are there in an over?", wrong_context)[0])

Passages given: 4.1, 5.7, 9.1
Answer: The context does not say.


In [21]:
changed_fact = chunks_df[chunks_df["clause"] == "40.1"].head(1).copy()
changed_fact["text"] = changed_fact["text"].str.replace("within 3 minutes", "within 5 minutes")
assert "within 5 minutes" in changed_fact["text"].iloc[0]
print("Answer:", generate(SANITY_QUERIES[2], changed_fact)[0])

Answer: The incoming batter must be ready within 5 minutes of the dismissal or retirement; if not, they are out Timed out. [1] "within 5 minutes of the dismissal or retirement. If this requirement is not met, the incoming batter will be out, Timed out."


## 6. Failure mode analysis

### 6.1 Screening candidate queries

Queries were written for each failure category from the brief. For each one, the clause(s) that actually
answer it ("gold") were looked up in the Laws. Retrieval and re-ranking are run (no LLM) and we record where
the best gold chunk lands:
- **bi** = its rank in the bi-encoder top 10 (">10" if it isn't retrieved at all)
- **ce** = its rank after cross-encoder re-ranking
- **verdict**: does each pipeline put a gold chunk in the top 3 that the LLM sees?

All candidates are kept here, including the ones where re-ranking made no difference or made things worse.

In [22]:
SCREEN_QUERIES = [
    # (category, query, gold clauses)
    ("keyword trap", "What happens if the ball hits a helmet lying on the ground behind the wicket-keeper?", ["28.3"]),
    ("keyword trap", "A dog runs onto the field and stops the ball before it reaches the rope. Is it four runs?", ["19.2", "20.4"]),
    ("keyword trap", "The ball cannot be found after a big hit. What happens?", ["4.5"]),
    ("keyword trap", "Can a fielder use their cap to stop the ball?", ["28.2"]),
    ("keyword trap", "Can the bowler Mankad the non-striker?", ["38.3"]),
    ("exception", "Can a batter be out caught off a no ball?", ["21.17"]),
    ("exception", "Can a batter be out LBW from a no ball?", ["21.17"]),
    ("exception", "Can the striker be stumped off a wide?", ["22.9", "39.4"]),
    ("exception", "Does the bowler get credit for a run out?", ["38.6"]),
    ("exception", "Is a bowler allowed to change ends whenever they want?", ["17.6"]),
    ("exception", "If the bowler overstepped, can the batter still be caught?", ["21.17"]),
    ("exception", "If the ball was too far outside off stump to reach, can the batter still be stumped?", ["22.9", "39.4"]),
    ("exception", "Is a catch fair if the fielder touches the boundary rope while holding the ball?", ["33.2"]),
    ("exception", "If the ball hits the batter's pad when they weren't trying to hit it, can they still run?", ["23.3"]),
    ("exception", "The ball hits the striker's helmet and is then caught by a fielder. Is that out?", ["33.1", "33.2"]),
    ("exception", "If the bowler deflects the ball onto the non-striker's stumps while the batter is out of their ground, is it out?", ["38.1", "38.2"]),
    ("exception", "A fielder catches the ball while standing outside the boundary. Is the batter out?", ["33.2", "19.4", "19.5"]),
    ("exception", "Can the batters run a bye off a wide?", ["23.1", "22.6"]),
    ("negation", "What is a substitute fielder not allowed to do?", ["24.1"]),
    ("negation", "Which players are not allowed to wear gloves?", ["28.1", "27.1"]),
    ("negation", "Can a bowler bowl two overs in a row?", ["17.6", "17.8"]),
    ("negation", "Who is not allowed to act as a runner?", ["25.1"]),
    ("negation", "When is short-pitched bowling not allowed?", ["41.6"]),
    ("negation", "Which deliveries is the bowler not allowed to bowl above waist height?", ["41.7"]),
    ("negation", "What may the fielding side not do to the ball?", ["41.3"]),
    ("negation", "When can byes not be scored?", ["23.1"]),
    ("negation", "When is the striker not allowed to be given out LBW?", ["36.1", "36.2"]),
    ("negation", "How many fielders are not allowed behind square on the leg side?", ["28.4"]),
    ("negation", "Which fielders are not allowed to stand on the pitch?", ["28.5"]),
    ("negation", "When can a batter not be given out even if the fielders appeal?", ["31.1", "31.2"]),
    ("negation", "What must a fielder not do while the striker is facing?", ["41.4"]),
    ("negation", "Which player is not allowed to come back on the field without the umpire's permission?", ["24.2", "24.4"]),
    ("negation", "When is the wicket-keeper not allowed to move?", ["27.3", "27.4"]),
    ("buried / slang", "Is it legal to run out a non-striker who backs up too far before the ball is bowled?", ["38.3"]),
    ("buried / slang", "What happens if a bouncer goes over the batter's head?", ["22.1", "41.6"]),
    ("buried / slang", "Can a batter be given out for hitting the ball a second time on purpose?", ["34.1"]),
    ("buried / slang", "What happens if the bowler bowls a beamer?", ["41.7"]),
    ("buried / slang", "What is the penalty for ball tampering?", ["41.3"]),
    ("buried / slang", "What happens if a bowler is caught chucking?", ["21.2", "21.3"]),
    ("buried / slang", "Can play stop because of bad light?", ["2.7", "2.8"]),
    ("buried / slang", "What happens to a batter who retires hurt?", ["25.4"]),
    ("buried / slang", "How are overthrows that reach the boundary scored?", ["19.8"]),
    ("buried / slang", "Is fake fielding allowed?", ["41.5"]),
    ("buried / slang", "Is sledging the batter allowed?", ["41.5", "42.2", "42.3"]),
    ("buried / slang", "What if the bowler oversteps the crease on purpose?", ["41.8"]),
    ("buried / slang", "What happens if the batters run one short?", ["18.3"]),
    # second round: everyday wording and slang
    ("lay / slang", "Is the batter out if they block the ball with their hand without holding the bat?", ["37.1"]),
    ("lay / slang", "How many runs if the ball clears the rope on the full?", ["19.7"]),
    ("lay / slang", "Is the keeper allowed to stand right up next to the stumps before the ball is bowled?", ["27.3"]),
    ("lay / slang", "Is the batter out if the ball hits the stumps but the bails stay on?", ["29.1", "29.2"]),
    ("lay / slang", "Can the fielding captain call back a batter after they were given out?", ["31.8"]),
    ("lay / slang", "Is it allowed to bowl underarm?", ["21.1"]),
    ("lay / slang", "What happens if a left-arm bowler switches to bowling right-arm without telling anyone?", ["21.1"]),
    ("lay / slang", "Can a team declare without batting at all?", ["15.2"]),
    ("lay / slang", "What happens if both teams finish with exactly the same score?", ["16.5", "16.1", "16.2"]),
    ("lay / slang", "Can the batting team change the time of the tea break?", ["11.6", "11.4"]),
    ("lay / slang", "What if the non-striker gets in the way of a fielder taking a catch?", ["37.3"]),
    ("lay / slang", "Is the batter out if they walk off thinking they were out when they weren't?", ["31.7"]),
    ("lay / slang", "Can the striker take guard on the pitch in front of the stumps and scuff it up?", ["41.14", "41.15"]),
    ("lay / slang", "What happens if the batter's bat breaks and a piece hits the stumps?", ["35.1", "35.2"]),
    ("lay / slang", "Is a batter out if the keeper drops the ball and it rebounds onto the stumps while the batter is out of the crease?", ["39.2"]),
    ("lay / slang", "Can a fielder field the ball with a towel or sweater?", ["28.2"]),
    ("lay / slang", "What happens if the bowler throws the ball at the batter's end before bowling it?", ["21.4"]),
    ("lay / slang", "Do the umpires have to agree before giving someone out run out at the far end?", ["31.6"]),
    ("lay / slang", "Can a batter switch from right-handed to left-handed as the bowler runs in?", ["25.8"]),
    ("lay / slang", "How many short balls per over can a bowler bowl before it's unfair?", ["41.6"]),
    ("lay / slang", "Can the twelfth man bowl?", ["24.1"]),
    ("lay / slang", "Can the twelfth man keep wicket?", ["24.1"]),
    ("lay / slang", "Is a batter out for handling the ball?", ["37.1"]),
    ("lay / slang", "Can a batter be out handled the ball if they pick it up off the ground?", ["37.1"]),
    ("lay / slang", "What happens if a batter retires without being injured?", ["25.4"]),
    ("lay / slang", "Is a batter out if they are caught off a bump ball?", ["33.2"]),
    ("lay / slang", "Is a grubber that rolls along the ground a legal delivery?", ["21.7"]),
    ("lay / slang", "What is the penalty for a slow over rate?", ["41.9"]),
    ("lay / slang", "Can a fielder shine the ball with saliva or sweat?", ["41.3"]),
    ("lay / slang", "Can a player polish the ball on their trousers?", ["41.3"]),
    ("lay / slang", "Can an injured batter get someone to run for them?", ["25.5"]),
    ("lay / slang", "Who decides whether the pitch is too wet to play on?", ["2.7"]),
    ("lay / slang", "Can the groundsman roll the pitch during the match?", ["9.1"]),
    ("lay / slang", "Can a bowler send down a bouncer every ball?", ["41.6"]),
    ("lay / slang", "Is it legal to bowl a full toss at the batter's head?", ["41.7"]),
    ("lay / slang", "Can the keeper stand in front of the stumps to take the ball?", ["27.3"]),
    ("lay / slang", "What happens when the batter gets a top edge that is caught by the keeper after hitting his helmet grille?", ["33.2"]),
    ("lay / slang", "Can the non-striker steal a single while the bowler is running in?", ["41.16"]),
    ("lay / slang", "Is a batter allowed to run down the middle of the pitch?", ["41.14", "41.15"]),
    ("lay / slang", "What happens when the umpire signals a dead ball because a fielder hurts themselves?", ["20.4"]),
]


def gold_ranks(query, gold):
    """Best rank of any gold chunk in the bi-encoder top 10 and after re-ranking (None = not retrieved)."""
    candidates, _ = retrieve_top10(query)
    ranked, _ = rerank(query, candidates)
    g = ranked[ranked["clause"].isin(gold)]
    if g.empty:
        return None, None
    return int(g["bi_rank"].min()), int(g["ce_rank"].min())


rows = []
for category, query, gold in SCREEN_QUERIES:
    bi, ce = gold_ranks(query, gold)
    base_ok = bi is not None and bi <= TOP_K_CONTEXT
    rr_ok = ce is not None and ce <= TOP_K_CONTEXT
    verdict = ("re-rank helps" if rr_ok and not base_ok else "re-rank hurts" if base_ok and not rr_ok
               else "both ok" if base_ok else "both miss")
    rows.append({"category": category, "query": query, "gold": ", ".join(gold),
                 "bi": bi or ">10", "ce": ce or ">10", "verdict": verdict})

screen_df = pd.DataFrame(rows)
print(screen_df["verdict"].value_counts().to_string(), "\n")
pd.set_option("display.max_colwidth", 90)
screen_df

verdict
both ok          46
both miss        26
re-rank hurts     8
re-rank helps     6 



,category,query,gold,bi,ce,verdict
0,keyword trap,What happens if the ball hits a helmet lying on the ground behind the wicket-keeper?,28.3,1,1,both ok
1,keyword trap,A dog runs onto the field and stops the ball before it reaches the rope. Is it four runs?,"19.2, 20.4",>10,>10,both miss
2,keyword trap,The ball cannot be found after a big hit. What happens?,4.5,1,1,both ok
3,keyword trap,Can a fielder use their cap to stop the ball?,28.2,1,2,both ok
4,keyword trap,Can the bowler Mankad the non-striker?,38.3,7,1,re-rank helps
...,...,...,...,...,...,...
81,lay / slang,Can the keeper stand in front of the stumps to take the ball?,27.3,2,4,re-rank hurts
82,lay / slang,What happens when the batter gets a top edge that is caught by the keeper after hittin...,33.2,6,3,re-rank helps
83,lay / slang,Can the non-striker steal a single while the bowler is running in?,41.16,1,1,both ok
84,lay / slang,Is a batter allowed to run down the middle of the pitch?,"41.14, 41.15",>10,>10,both miss


**What the screening shows**
- For most negation and exception queries, the bi-encoder already ranks the right clause #1. Every chunk starts
  with its Law and clause title (e.g. "Out from a No ball", "Fielders not to encroach on pitch"), and those titles
  share many words with such questions.
- Re-ranking helps most when the query uses **different words from the Laws**: cricket slang ("Mankad", "beamer",
  "bad light") that never appears in the text. The bi-encoder then ranks chunks with overlapping words higher, and the
  cross-encoder, which reads query and chunk together, recognises the matching rule.
- In a few cases re-ranking **hurts**: the right chunk was in the baseline top 3 but the cross-encoder pushed it out.
- Some queries are missed by both pipelines because the right chunk isn't in the top 10 at all; re-ranking cannot
  recover a chunk the bi-encoder never retrieved.

Below: five cases where re-ranking helps, then two where it hurts. Each has the full top-10 rank table
and both LLM answers.

### 6.2 Keyword trap: "Mankad"

**Query:** Can the bowler Mankad the non-striker?

"Mankad" (running out the non-striker before the ball is bowled) is slang that never appears in the Laws. The query words *bowler* and *non-striker* match other clauses, e.g. 30.3 (Position of non-striker) and 21.4 (Bowler throwing towards striker's end before delivery). The answer is in **38.3 Non-striker leaving their ground early**, which uses none of the query's distinctive words.

In [23]:
q = "Can the bowler Mankad the non-striker?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,278,Law 30,30.3 Position of non-striker,1,0.747,2,3.21,#1 → #2,both
1,287,Law 32,32.1 Out Bowled,2,0.730,3,2.65,#2 → #3,both
2,192,Law 21,21.4 Bowler throwing towards striker’s end before delivery,3,0.729,4,-1.19,#3 → #4,baseline
3,243,Law 25,25.8 Striker’s right to play the ball,4,0.726,9,-6.62,#4 → #9,
4,256,Law 27,27.6 Interference with wicket-keeper by striker,5,0.724,7,-5.15,#5 → #7,
5,196,Law 21,"21.8 Ball causing the striker to leave the pitch, or ball coming to rest in front of s...",6,0.718,6,-3.02,#6 → #6,
6,318,Law 38,38.3 Non-striker leaving their ground early,7,0.718,1,3.70,#7 → #1,re-rank
7,302,Law 35,35.2 Not out Hit wicket,8,0.712,10,-7.44,#8 → #10,
8,297,Law 34,34.3 Ball lawfully struck more than once,9,0.711,8,-5.82,#9 → #8,
9,305,Law 36,36.1 Out LBW,10,0.710,5,-2.90,#10 → #5,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,30.3 Position of non-striker,0.747,38.3 Non-striker leaving their ground early,3.70,7
2,32.1 Out Bowled,0.730,30.3 Position of non-striker,3.21,1
3,21.4 Bowler throwing towards striker’s end before delivery,0.729,32.1 Out Bowled,2.65,2
4,25.8 Striker’s right to play the ball,0.726,21.4 Bowler throwing towards striker’s end before delivery,-1.19,3
5,27.6 Interference with wicket-keeper by striker,0.724,36.1 Out LBW,-2.90,10
6,"21.8 Ball causing the striker to leave the pitch, or ball coming to rest in front of s...",0.718,"21.8 Ball causing the striker to leave the pitch, or ball coming to rest in front of s...",-3.02,6
7,38.3 Non-striker leaving their ground early,0.718,27.6 Interference with wicket-keeper by striker,-5.15,5
8,35.2 Not out Hit wicket,0.712,34.3 Ball lawfully struck more than once,-5.82,9
9,34.3 Ball lawfully struck more than once,0.711,25.8 Striker’s right to play the ball,-6.62,4


In [24]:
p1, p2 = run_both(q)

QUESTION: Can the bowler Mankad the non-striker?

Pipeline 1 (baseline) top 3:
  bi #1  Law 30 – Batter out of their ground | 30.3 Position of non-striker
  bi #2  Law 32 – Bowled | 32.1 Out Bowled
  bi #3  Law 21 – No ball | 21.4 Bowler throwing towards striker’s end before delivery
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #7)  Law 38 – Run out | 38.3 Non-striker leaving their ground early
  ce #2 (was bi #1)  Law 30 – Batter out of their ground | 30.3 Position of non-striker
  ce #3 (was bi #2)  Law 32 – Bowled | 32.1 Out Bowled

ANSWER 1: The context does not say.

ANSWER 2: “Mankad” refers to running out the non-striker for leaving their ground early, which is covered by [1] Law 38.3.

Yes, but only within a limited window. Under [1] 38.3.1, if the non-striker is out of their ground from the moment the ball comes into play until the bowler’s arm reaches the highest point of their normal bowling action, they are out Run out if the wicket is broken by the bowler throwing the bal

### 6.3 Buried chunk: "beamer"

**Query:** What happens if the bowler bowls a beamer?

A beamer is a full toss above waist height. The Laws call it a *non-landing delivery* (**41.7**), and the related short-pitched rule is **41.6**. The bi-encoder ranks general bowler clauses (changing ends, running on the protected area) above them.

In [25]:
q = "What happens if the bowler bowls a beamer?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,138,Law 17,17.6 Bowler changing ends,1,0.667,8,-5.78,#1 → #8,baseline
1,361,Law 41,41.13 Bowler running on protected area,2,0.664,4,-4.69,#2 → #4,baseline
2,350,Law 41,41.8 Bowling of deliberate front-foot No ball,3,0.663,1,-4.02,#3 → #1,both
3,194,Law 21,21.6 Bowler breaking wicket in delivering ball,4,0.656,7,-5.32,#4 → #7,
4,348,Law 41,41.7 Bowling of dangerous and unfair non-landing deliveries,5,0.652,2,-4.58,#5 → #2,re-rank
5,140,Law 17,17.8 Bowler incapacitated or suspended during an over,6,0.646,5,-4.70,#6 → #5,
6,299,Law 34,34.5 Bowler does not get credit,7,0.646,10,-8.80,#7 → #10,
7,344,Law 41,41.6 Bowling of dangerous and unfair short deliveries,8,0.643,3,-4.65,#8 → #3,re-rank
8,336,Law 41,41.3 The match ball – changing its condition,9,0.642,9,-5.92,#9 → #9,
9,412,Appendix A,A.9 Bowlers,10,0.641,6,-5.18,#10 → #6,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,17.6 Bowler changing ends,0.667,41.8 Bowling of deliberate front-foot No ball,-4.02,3
2,41.13 Bowler running on protected area,0.664,41.7 Bowling of dangerous and unfair non-landing deliveries,-4.58,5
3,41.8 Bowling of deliberate front-foot No ball,0.663,41.6 Bowling of dangerous and unfair short deliveries,-4.65,8
4,21.6 Bowler breaking wicket in delivering ball,0.656,41.13 Bowler running on protected area,-4.69,2
5,41.7 Bowling of dangerous and unfair non-landing deliveries,0.652,17.8 Bowler incapacitated or suspended during an over,-4.70,6
6,17.8 Bowler incapacitated or suspended during an over,0.646,A.9 Bowlers,-5.18,10
7,34.5 Bowler does not get credit,0.646,21.6 Bowler breaking wicket in delivering ball,-5.32,4
8,41.6 Bowling of dangerous and unfair short deliveries,0.643,17.6 Bowler changing ends,-5.78,1
9,41.3 The match ball – changing its condition,0.642,41.3 The match ball – changing its condition,-5.92,9


In [26]:
p1, p2 = run_both(q)

QUESTION: What happens if the bowler bowls a beamer?

Pipeline 1 (baseline) top 3:
  bi #1  Law 17 – The over | 17.6 Bowler changing ends
  bi #2  Law 41 – Unfair play | 41.13 Bowler running on protected area
  bi #3  Law 41 – Unfair play | 41.8 Bowling of deliberate front-foot No ball
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #3)  Law 41 – Unfair play | 41.8 Bowling of deliberate front-foot No ball
  ce #2 (was bi #5)  Law 41 – Unfair play | 41.7 Bowling of dangerous and unfair non-landing deliveries
  ce #3 (was bi #8)  Law 41 – Unfair play | 41.6 Bowling of dangerous and unfair short deliveries

ANSWER 1: The context does not say.

ANSWER 2: A beamer is a non-landing delivery, so it matches passage [2] (Law 41.7, dangerous and unfair non-landing deliveries). The context only covers a deliberate beamer: if the umpire considers the bowler deliberately bowled a non-landing delivery deemed unfair, the caution and warning are dispensed with; the umpire immediately calls and signals N

### 6.4 Synonym: "bad light"

**Query:** Can play stop because of bad light?

The Laws cover light under **2.7 Fitness for play** ("conditions of ground, weather or light"). The bi-encoder prefers clauses about stopping/concluding play and intervals.

In [27]:
q = "Can play stop because of bad light?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,103,Law 12,12.9 Conclusion of match,1,0.621,2,-2.88,#1 → #2,both
1,86,Law 11,11.6 Changing agreed time for tea interval,2,0.584,1,-1.20,#2 → #1,both
2,99,Law 12,12.7 Last hour of match – interruptions of play,3,0.571,8,-10.21,#3 → #8,baseline
3,104,Law 12,12.10 Completion of last over of match,4,0.558,4,-9.08,#4 → #4,
4,362,Law 41,41.14 Batter damaging the pitch,5,0.556,5,-10.02,#5 → #5,
5,358,Law 41,41.12 Fielder damaging the pitch,6,0.555,9,-10.36,#6 → #9,
6,13,Law 2,2.7 Fitness for play,7,0.554,3,-5.89,#7 → #3,re-rank
7,183,Law 20,20.5 Ball ceases to be dead,8,0.552,7,-10.18,#8 → #7,
8,76,Law 10,10.3 Removal of covers,9,0.552,10,-10.99,#9 → #10,
9,92,Law 12,12.1 Call of Play,10,0.550,6,-10.06,#10 → #6,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,12.9 Conclusion of match,0.621,11.6 Changing agreed time for tea interval,-1.20,2
2,11.6 Changing agreed time for tea interval,0.584,12.9 Conclusion of match,-2.88,1
3,12.7 Last hour of match – interruptions of play,0.571,2.7 Fitness for play,-5.89,7
4,12.10 Completion of last over of match,0.558,12.10 Completion of last over of match,-9.08,4
5,41.14 Batter damaging the pitch,0.556,41.14 Batter damaging the pitch,-10.02,5
6,41.12 Fielder damaging the pitch,0.555,12.1 Call of Play,-10.06,10
7,2.7 Fitness for play,0.554,20.5 Ball ceases to be dead,-10.18,8
8,20.5 Ball ceases to be dead,0.552,12.7 Last hour of match – interruptions of play,-10.21,3
9,10.3 Removal of covers,0.552,41.12 Fielder damaging the pitch,-10.36,6


In [28]:
p1, p2 = run_both(q)

QUESTION: Can play stop because of bad light?

Pipeline 1 (baseline) top 3:
  bi #1  Law 12 – Start of play; cessation of play | 12.9 Conclusion of match
  bi #2  Law 11 – Intervals | 11.6 Changing agreed time for tea interval
  bi #3  Law 12 – Start of play; cessation of play | 12.7 Last hour of match – interruptions of play
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #2)  Law 11 – Intervals | 11.6 Changing agreed time for tea interval
  ce #2 (was bi #1)  Law 12 – Start of play; cessation of play | 12.9 Conclusion of match
  ce #3 (was bi #7)  Law 2 – The umpires | 2.7 Fitness for play

ANSWER 1: Yes. [1] "the players leave the field for adverse conditions of ground, weather or light, or in exceptional circumstances, and no further play is possible." [2] also allows a stoppage "because of adverse conditions of ground, weather or light."

ANSWER 2: Yes. The umpires together decide whether conditions of light make play dangerous or unreasonable [3] "It is solely for the umpires toget

### 6.5 Slang: "grubber"

**Query:** Is a grubber that rolls along the ground a legal delivery?

A grubber is a delivery that rolls along the ground. The rule is **21.7 Ball bouncing more than once, rolling along the ground or landing off the pitch** (it is a No ball). The bi-encoder ranks the general "fair delivery" clauses (21.5 feet, 21.2 arm) higher.

In [29]:
q = "Is a grubber that rolls along the ground a legal delivery?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,193,Law 21,21.5 Fair delivery – the feet,1,0.637,5,-7.85,#1 → #5,baseline
1,188,Law 21,21.2 Fair delivery – the arm,2,0.634,4,-7.85,#2 → #4,baseline
2,345,Law 41,41.7 Bowling of dangerous and unfair non-landing deliveries,3,0.619,6,-8.84,#3 → #6,baseline
3,325,Law 39,39.3 Not out Stumped,4,0.617,3,-7.75,#4 → #3,re-rank
4,195,Law 21,"21.7 Ball bouncing more than once, rolling along the ground or landing off the pitch",5,0.606,1,0.27,#5 → #1,re-rank
5,72,Law 9,9.7 Securing of footholds and maintenance of pitch,6,0.606,10,-11.44,#6 → #10,
6,275,Law 30,30.1 When out of their ground,7,0.602,8,-10.50,#7 → #8,
7,192,Law 21,21.4 Bowler throwing towards striker’s end before delivery,8,0.600,2,-7.37,#8 → #2,re-rank
8,209,Law 22,22.4 Delivery not a Wide,9,0.599,7,-9.48,#9 → #7,
9,342,Law 41,41.6 Bowling of dangerous and unfair short deliveries,10,0.596,9,-11.06,#10 → #9,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,21.5 Fair delivery – the feet,0.637,"21.7 Ball bouncing more than once, rolling along the ground or landing off the pitch",0.27,5
2,21.2 Fair delivery – the arm,0.634,21.4 Bowler throwing towards striker’s end before delivery,-7.37,8
3,41.7 Bowling of dangerous and unfair non-landing deliveries,0.619,39.3 Not out Stumped,-7.75,4
4,39.3 Not out Stumped,0.617,21.2 Fair delivery – the arm,-7.85,2
5,"21.7 Ball bouncing more than once, rolling along the ground or landing off the pitch",0.606,21.5 Fair delivery – the feet,-7.85,1
6,9.7 Securing of footholds and maintenance of pitch,0.606,41.7 Bowling of dangerous and unfair non-landing deliveries,-8.84,3
7,30.1 When out of their ground,0.602,22.4 Delivery not a Wide,-9.48,9
8,21.4 Bowler throwing towards striker’s end before delivery,0.600,30.1 When out of their ground,-10.50,7
9,22.4 Delivery not a Wide,0.599,41.6 Bowling of dangerous and unfair short deliveries,-11.06,10


In [30]:
p1, p2 = run_both(q)

QUESTION: Is a grubber that rolls along the ground a legal delivery?

Pipeline 1 (baseline) top 3:
  bi #1  Law 21 – No ball | 21.5 Fair delivery – the feet
  bi #2  Law 21 – No ball | 21.2 Fair delivery – the arm
  bi #3  Law 41 – Unfair play | 41.7 Bowling of dangerous and unfair non-landing deliveries
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #5)  Law 21 – No ball | 21.7 Ball bouncing more than once, rolling along the ground or landing off the pitch
  ce #2 (was bi #8)  Law 21 – No ball | 21.4 Bowler throwing towards striker’s end before delivery
  ce #3 (was bi #4)  Law 39 – Stumped | 39.3 Not out Stumped

ANSWER 1: The context does not say.

ANSWER 2: A grubber is a delivery that rolls along the ground, which matches Law 21.7.1 in passage [1]. It is not a legal delivery in that case: [1] “The umpire shall call and signal No ball if a ball which they consider to have been delivered, without having previously touched bat or person of the striker, … 21.7.1 … rolls along the groun

### 6.6 Slang: "slow over rate"

**Query:** What is the penalty for a slow over rate?

"Over rate" never appears in the Laws. The rule is **41.9 Time wasting by the fielding side**. The word *penalty* pulls the bi-encoder towards penalty clauses for Wides and penalty runs in general.

In [31]:
q = "What is the penalty for a slow over rate?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,211,Law 22,22.6 Penalty for a Wide,1,0.643,3,-7.30,#1 → #3,both
1,372,Law 41,41.17 Penalty runs,2,0.636,8,-8.74,#2 → #8,baseline
2,149,Law 18,18.6 Runs awarded for penalties,3,0.634,6,-8.66,#3 → #6,baseline
3,353,Law 41,41.9 Time wasting by the fielding side,4,0.630,2,-5.32,#4 → #2,re-rank
4,170,Law 19,19.7 Runs scored from boundaries,5,0.624,10,-10.11,#5 → #10,
5,370,Law 41,41.17 Penalty runs,6,0.621,4,-7.36,#6 → #4,
6,226,Law 24,24.3 Penalty time not incurred,7,0.621,9,-9.84,#7 → #9,
7,371,Law 41,41.17 Penalty runs,8,0.615,7,-8.67,#8 → #7,
8,352,Law 41,41.9 Time wasting by the fielding side,9,0.614,1,-2.65,#9 → #1,re-rank
9,383,Law 42,42.4 Level 3 offences and action by umpires,10,0.611,5,-8.27,#10 → #5,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,22.6 Penalty for a Wide,0.643,41.9 Time wasting by the fielding side,-2.65,9
2,41.17 Penalty runs,0.636,41.9 Time wasting by the fielding side,-5.32,4
3,18.6 Runs awarded for penalties,0.634,22.6 Penalty for a Wide,-7.30,1
4,41.9 Time wasting by the fielding side,0.630,41.17 Penalty runs,-7.36,6
5,19.7 Runs scored from boundaries,0.624,42.4 Level 3 offences and action by umpires,-8.27,10
6,41.17 Penalty runs,0.621,18.6 Runs awarded for penalties,-8.66,3
7,24.3 Penalty time not incurred,0.621,41.17 Penalty runs,-8.67,8
8,41.17 Penalty runs,0.615,41.17 Penalty runs,-8.74,2
9,41.9 Time wasting by the fielding side,0.614,24.3 Penalty time not incurred,-9.84,7


In [32]:
p1, p2 = run_both(q)

QUESTION: What is the penalty for a slow over rate?

Pipeline 1 (baseline) top 3:
  bi #1  Law 22 – Wide ball | 22.6 Penalty for a Wide
  bi #2  Law 41 – Unfair play | 41.17 Penalty runs
  bi #3  Law 18 – Scoring runs | 18.6 Runs awarded for penalties
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #9)  Law 41 – Unfair play | 41.9 Time wasting by the fielding side
  ce #2 (was bi #4)  Law 41 – Unfair play | 41.9 Time wasting by the fielding side
  ce #3 (was bi #1)  Law 22 – Wide ball | 22.6 Penalty for a Wide

ANSWER 1: The context does not say.

ANSWER 2: “Slow over rate” matches Law 41.9 on time wasting by the fielding side / unnecessarily slow progress of an over in [1] and [2].

First instance: the umpire calls Dead ball, and the bowler’s end umpire warns the captain of the fielding side that it is a first and final warning, and informs the batters. [1] “call and signal Dead ball”; [1] “warn the captain of the fielding side, indicating that this is a first and final warning”; [1] “i

### 6.7 Counterexample (negation): re-ranking hurts

**Query:** When can byes not be scored?

The answer is in **23.1 Byes** (byes only count when the ball is not a Wide and passes the striker without touching their bat or person). The bi-encoder has it at #2; the cross-encoder prefers 23.3 *Leg byes not to be awarded*, whose title matches "not ... scored" but is about leg byes, and pushes 23.1 out of the top 3.

In [33]:
q = "When can byes not be scored?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,149,Law 18,18.6 Runs awarded for penalties,1,0.746,4,2.44,#1 → #4,baseline
1,215,Law 23,23.1 Byes,2,0.745,6,1.88,#2 → #6,baseline
2,217,Law 23,23.2 Leg byes,3,0.741,3,2.94,#3 → #3,both
3,218,Law 23,23.3 Leg byes not to be awarded,4,0.741,1,6.16,#4 → #1,re-rank
4,142,Law 18,18.2 Runs disallowed and runs not scored,5,0.740,7,0.16,#5 → #7,
5,293,Law 33,33.4 No runs to be scored,6,0.739,9,-0.94,#6 → #9,
6,415,Appendix A,A.11 Runs,7,0.737,8,-0.92,#7 → #8,
7,203,Law 21,21.15 Runs resulting from a No ball – how scored,8,0.734,5,2.06,#8 → #5,
8,313,Law 37,37.5 Runs scored,9,0.728,10,-4.27,#9 → #10,
9,152,Law 18,18.9 Runs scored when the ball becomes dead other than at the fall of a wicket,10,0.715,2,3.27,#10 → #2,re-rank


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,18.6 Runs awarded for penalties,0.746,23.3 Leg byes not to be awarded,6.16,4
2,23.1 Byes,0.745,18.9 Runs scored when the ball becomes dead other than at the fall of a wicket,3.27,10
3,23.2 Leg byes,0.741,23.2 Leg byes,2.94,3
4,23.3 Leg byes not to be awarded,0.741,18.6 Runs awarded for penalties,2.44,1
5,18.2 Runs disallowed and runs not scored,0.740,21.15 Runs resulting from a No ball – how scored,2.06,8
6,33.4 No runs to be scored,0.739,23.1 Byes,1.88,2
7,A.11 Runs,0.737,18.2 Runs disallowed and runs not scored,0.16,5
8,21.15 Runs resulting from a No ball – how scored,0.734,A.11 Runs,-0.92,7
9,37.5 Runs scored,0.728,33.4 No runs to be scored,-0.94,6


In [34]:
p1, p2 = run_both(q)

QUESTION: When can byes not be scored?

Pipeline 1 (baseline) top 3:
  bi #1  Law 18 – Scoring runs | 18.6 Runs awarded for penalties
  bi #2  Law 23 – Bye and leg bye | 23.1 Byes
  bi #3  Law 23 – Bye and leg bye | 23.2 Leg byes
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #4)  Law 23 – Bye and leg bye | 23.3 Leg byes not to be awarded
  ce #2 (was bi #10)  Law 18 – Scoring runs | 18.9 Runs scored when the ball becomes dead other than at the fall of a wicket
  ce #3 (was bi #3)  Law 23 – Bye and leg bye | 23.2 Leg byes

ANSWER 1: Byes cannot be scored when the delivery is a Wide, or when the ball touches the striker’s bat or person, because byes are only credited when the ball is not a Wide and passes the striker without touching the bat or person. [2] "not being a Wide" and "without touching the striker’s bat or person"

ANSWER 2: The context does not say.


### 6.8 Counterexample (exception): re-ranking hurts

**Query:** Can a batter be given out for hitting the ball a second time on purpose?

The main rule is **34.1 Out Hit the ball twice**; the exceptions are in 34.2/34.3. The bi-encoder ranks 34.1 #1. The cross-encoder moves the *Not out* clauses (37.2, 34.2) to the top and 34.1 drops out of the top 3.

In [35]:
q = "Can a batter be given out for hitting the ball a second time on purpose?"
display(rank_table(q))
side_by_side(q)

,chunk_id,law,chunk,bi_rank,bi_score,ce_rank,ce_score,shift,used_by
0,295,Law 34,34.1 Out Hit the ball twice,1,0.780,5,1.53,#1 → #5,baseline
1,297,Law 34,34.3 Ball lawfully struck more than once,2,0.777,3,3.35,#2 → #3,both
2,298,Law 34,34.4 Runs permitted from ball lawfully struck more than once,3,0.771,8,-0.43,#3 → #8,baseline
3,296,Law 34,34.2 Not out Hit the ball twice,4,0.764,2,4.52,#4 → #2,re-rank
4,285,Law 31,31.7 Batter leaving the wicket under a misapprehension,5,0.735,6,1.40,#5 → #6,
5,308,Law 37,37.1 Out Obstructing the field,6,0.733,4,2.03,#6 → #4,
6,310,Law 37,37.2 Not out Obstructing the field,7,0.728,1,4.90,#7 → #1,re-rank
7,339,Law 41,"41.5 Deliberate distraction, deception or obstruction of batter",8,0.721,10,-4.81,#8 → #10,
8,291,Law 33,33.2 A fair catch,9,0.717,9,-1.00,#9 → #9,
9,205,Law 21,21.17 Out from a No ball,10,0.712,7,1.38,#10 → #7,


,bi-encoder order,bi score,cross-encoder order,ce score,was bi #
rank,,,,,
1,34.1 Out Hit the ball twice,0.780,37.2 Not out Obstructing the field,4.90,7
2,34.3 Ball lawfully struck more than once,0.777,34.2 Not out Hit the ball twice,4.52,4
3,34.4 Runs permitted from ball lawfully struck more than once,0.771,34.3 Ball lawfully struck more than once,3.35,2
4,34.2 Not out Hit the ball twice,0.764,37.1 Out Obstructing the field,2.03,6
5,31.7 Batter leaving the wicket under a misapprehension,0.735,34.1 Out Hit the ball twice,1.53,1
6,37.1 Out Obstructing the field,0.733,31.7 Batter leaving the wicket under a misapprehension,1.40,5
7,37.2 Not out Obstructing the field,0.728,21.17 Out from a No ball,1.38,10
8,"41.5 Deliberate distraction, deception or obstruction of batter",0.721,34.4 Runs permitted from ball lawfully struck more than once,-0.43,3
9,33.2 A fair catch,0.717,33.2 A fair catch,-1.00,9


In [36]:
p1, p2 = run_both(q)

QUESTION: Can a batter be given out for hitting the ball a second time on purpose?

Pipeline 1 (baseline) top 3:
  bi #1  Law 34 – Hit the ball twice | 34.1 Out Hit the ball twice
  bi #2  Law 34 – Hit the ball twice | 34.3 Ball lawfully struck more than once
  bi #3  Law 34 – Hit the ball twice | 34.4 Runs permitted from ball lawfully struck more than once
Pipeline 2 (re-ranked) top 3:
  ce #1 (was bi #7)  Law 37 – Obstructing the field | 37.2 Not out Obstructing the field
  ce #2 (was bi #4)  Law 34 – Hit the ball twice | 34.2 Not out Hit the ball twice
  ce #3 (was bi #2)  Law 34 – Hit the ball twice | 34.3 Ball lawfully struck more than once

ANSWER 1: Yes, if the striker wilfully strikes the ball again before a fielder touches it, unless it is solely to guard their wicket. [1] "The striker is out Hit the ball twice if... the striker wilfully strikes it again with their bat or person... except for the sole purpose of guarding their wicket."

ANSWER 2: The context does not contain t

### 6.9 Extra: can the failures be reduced?

Two experiments on the same screening queries (retrieval and re-ranking only, no LLM). The main pipelines above stay
as the brief specifies (top 10, `ms-marco-MiniLM-L-6-v2`).

1. **Re-rank a wider pool.** In the "both miss" cases the right chunk isn't in the bi-encoder's top 10, so no re-ranker
   can recover it. Does re-ranking the top 20 or 30 instead help?
2. **A larger cross-encoder.** `ms-marco-MiniLM-L-12-v2` has the same training data (web search) but twice the layers.
   Does it fix the "re-rank hurts" cases?

The baseline (bi-encoder top 3) is the same in every row, so only the re-ranked side changes.

In [37]:
def screen_with(reranker, pool_k=TOP_K_RETRIEVE):
    """Gold ranks for every screening query when `reranker` re-sorts the bi-encoder's top `pool_k`."""
    rows, ce_seconds = [], 0.0
    for category, query, gold in SCREEN_QUERIES:
        q_vec = bi_encoder.encode(QUERY_PREFIX + query, normalize_embeddings=True, convert_to_numpy=True)
        pool = np.argsort(-(chunk_embeddings @ q_vec))[:pool_k]
        clauses = chunks_df["clause"].values[pool]
        t0 = time.perf_counter()
        ce = reranker.predict([(query, chunks_df["text"].values[i]) for i in pool], show_progress_bar=False)
        ce_seconds += time.perf_counter() - t0
        bi_rank = next((r + 1 for r, c in enumerate(clauses) if c in gold), None)
        ce_rank = next((r + 1 for r, j in enumerate(np.argsort(-ce)) if clauses[j] in gold), None)
        base_ok = bi_rank is not None and bi_rank <= TOP_K_CONTEXT
        rr_ok = ce_rank is not None and ce_rank <= TOP_K_CONTEXT
        verdict = ("re-rank helps" if rr_ok and not base_ok else "re-rank hurts" if base_ok and not rr_ok
                   else "both ok" if base_ok else "both miss")
        rows.append({"query": query, "bi": bi_rank, "ce": ce_rank, "verdict": verdict})
    return pd.DataFrame(rows), ce_seconds / len(SCREEN_QUERIES)


def summarise(name, pool_k, df, secs):
    counts = df["verdict"].value_counts()
    return {"re-ranker": name, "pool": pool_k,
            "gold in pool": int(df["bi"].notna().sum()),
            **{v: int(counts.get(v, 0)) for v in ["re-rank helps", "re-rank hurts", "both ok", "both miss"]},
            "ms per query (cross-encoder)": round(secs * 1000)}


experiments = {}
for k in (10, 20, 30):
    experiments[("L-6", k)] = screen_with(cross_encoder, k)

pd.DataFrame([summarise(name, k, *res) for (name, k), res in experiments.items()])

,re-ranker,pool,gold in pool,re-rank helps,re-rank hurts,both ok,both miss,ms per query (cross-encoder)
0,L-6,10,68,6,8,46,26,327
1,L-6,20,75,5,8,46,27,611
2,L-6,30,77,7,8,46,25,1006


In [38]:
# Larger cross-encoder: same training data, 12 layers instead of 6 (one-time download, ~130 MB)
cross_encoder_l12 = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")
for k in (10, 30):
    experiments[("L-12", k)] = screen_with(cross_encoder_l12, k)

summary_df = pd.DataFrame([summarise(name, k, *res) for (name, k), res in experiments.items()])
summary_df

,re-ranker,pool,gold in pool,re-rank helps,re-rank hurts,both ok,both miss,ms per query (cross-encoder)
0,L-6,10,68,6,8,46,26,327
1,L-6,20,75,5,8,46,27,611
2,L-6,30,77,7,8,46,25,1006
3,L-12,10,68,7,7,47,25,1176
4,L-12,30,77,7,8,46,25,3203


In [39]:
# The queries where the L-6 re-ranker hurt (top-10 pool): does the L-12 model keep the right chunk in the top 3?
base = experiments[("L-6", 10)][0]
hurt = base[base["verdict"] == "re-rank hurts"][["query", "bi", "ce"]].rename(columns={"ce": "ce L-6"})
hurt["ce L-12"] = experiments[("L-12", 10)][0].loc[hurt.index, "ce"].values
hurt

,query,bi,ce L-6,ce L-12
11,"If the ball was too far outside off stump to reach, can the batter still be stumped?",1.0,5.0,5.0
15,If the bowler deflects the ball onto the non-striker's stumps while the batter is out ...,3.0,5.0,6.0
25,When can byes not be scored?,2.0,6.0,5.0
35,Can a batter be given out for hitting the ball a second time on purpose?,1.0,5.0,6.0
42,Is fake fielding allowed?,3.0,6.0,2.0
69,Can a batter be out handled the ball if they pick it up off the ground?,3.0,7.0,5.0
81,Can the keeper stand in front of the stumps to take the ball?,2.0,4.0,3.0
85,What happens when the umpire signals a dead ball because a fielder hurts themselves?,2.0,5.0,5.0


**Results of the two experiments** (86 screening queries)

| Setting | gold in pool | helps | hurts | both miss | cross-encoder time per query |
|---|---|---|---|---|---|
| L-6, top 10 (main pipeline) | 68 | 6 | 8 | 26 | ~330 ms |
| L-6, top 20 | 75 | 5 | 8 | 27 | ~610 ms |
| L-6, top 30 | 77 | 7 | 8 | 25 | ~1010 ms |
| L-12, top 10 | 68 | 7 | 7 | 25 | ~1180 ms |
| L-12, top 30 | 77 | 7 | 8 | 25 | ~3200 ms |

- A wider pool brings the right chunk into reach for 9 more queries (68 → 77), but the cross-encoder lifts almost none of
  them into the top 3, while its cost grows with the number of pairs (10 → 30 pairs took about 3× as long).
- The 12-layer cross-encoder is several times slower and changes the outcome for only a couple of queries: it fixes
  "fake fielding" and "keeper in front of the stumps", but the other "hurts" cases stay wrong.
- So on this document neither change is a cheap fix: the remaining failures come from the ms-marco models' training
  (web search, not rulebooks) and from the bi-encoder missing the right chunk entirely.

(The counts are the same on every run; the times depend on how busy the machine is.)

### 6.10 Metrics over all screening queries

The examples above show *how* re-ranking can help or hurt. These numbers show *how often*, over all 86 queries
(main pipeline: top 10, `ms-marco-MiniLM-L-6-v2`):
- **Hit rate @k**: share of queries with a gold chunk in the top k. @3 is what the LLM actually sees.
- **MRR** (mean reciprocal rank): average of 1/rank of the first gold chunk (0 if it isn't in the top 10). Higher is better.
- **Recall @10**: share of queries where a gold chunk is in the bi-encoder's top 10 at all. Re-ranking can't change this.

In [40]:
def retrieval_metrics(df):
    """df has one row per query with columns bi / ce = rank of the first gold chunk (NaN if not in the pool)."""
    out = {}
    for name, col in [("Baseline (bi-encoder)", "bi"), ("Re-ranked (cross-encoder)", "ce")]:
        r = df[col]
        out[name] = {"hit @1": (r <= 1).mean(), "hit @3": (r <= 3).mean(),
                     "MRR": (1 / r).fillna(0).mean(), "recall @10": r.notna().mean()}
    return pd.DataFrame(out).T.round(3)


main_screen = experiments[("L-6", 10)][0]
retrieval_metrics(main_screen)

,hit @1,hit @3,MRR,recall @10
Baseline (bi-encoder),0.477,0.628,0.572,0.791
Re-ranked (cross-encoder),0.488,0.605,0.571,0.791


### 6.11 Why does re-ranking add so little here? Removing the chunk headers

Every chunk starts with its Law and clause title (e.g. "Law 21 – No ball | 21.17 Out from a No ball"). These titles
summarise the rule in a few words, which gives the bi-encoder a strong hint. To test how much they matter, the same
screening is repeated on chunks **without** the header line: the chunks are re-embedded and the cross-encoder also sees
the header-free text. (This is a separate copy; the main index and pipelines are unchanged.)

In [41]:
def screen_on(texts, embeddings, reranker=cross_encoder, pool_k=TOP_K_RETRIEVE):
    """Same as screen_with, but on an alternative version of the chunk texts and embeddings."""
    rows = []
    for category, query, gold in SCREEN_QUERIES:
        q_vec = bi_encoder.encode(QUERY_PREFIX + query, normalize_embeddings=True, convert_to_numpy=True)
        pool = np.argsort(-(embeddings @ q_vec))[:pool_k]
        clauses = chunks_df["clause"].values[pool]
        ce = reranker.predict([(query, texts[i]) for i in pool], show_progress_bar=False)
        bi_rank = next((r + 1 for r, c in enumerate(clauses) if c in gold), None)
        ce_rank = next((r + 1 for r, j in enumerate(np.argsort(-ce)) if clauses[j] in gold), None)
        base_ok = bi_rank is not None and bi_rank <= TOP_K_CONTEXT
        rr_ok = ce_rank is not None and ce_rank <= TOP_K_CONTEXT
        verdict = ("re-rank helps" if rr_ok and not base_ok else "re-rank hurts" if base_ok and not rr_ok
                   else "both ok" if base_ok else "both miss")
        rows.append({"query": query, "bi": bi_rank, "ce": ce_rank, "verdict": verdict})
    return pd.DataFrame(rows)


# Drop the first line (the header) of every chunk and re-embed
texts_no_header = chunks_df["text"].str.split("\n", n=1).str[1].tolist()
t0 = time.perf_counter()
emb_no_header = bi_encoder.encode(texts_no_header, batch_size=32, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
print(f"Re-embedded {len(texts_no_header)} header-free chunks in {time.perf_counter() - t0:.1f} s")

no_header_screen = screen_on(texts_no_header, emb_no_header)

compare = pd.concat({"with headers": retrieval_metrics(main_screen),
                     "without headers": retrieval_metrics(no_header_screen)})
print(pd.DataFrame({"with headers": main_screen["verdict"].value_counts(),
                    "without headers": no_header_screen["verdict"].value_counts()}).fillna(0).astype(int), "\n")
compare

Re-embedded 432 header-free chunks in 15.8 s


               with headers  without headers
verdict                                     
both ok                  46               44
both miss                26               27
re-rank hurts             8                8
re-rank helps             6                7 



hit @1  hit @3    MRR  recall @10
with headers    Baseline (bi-encoder)       0.477   0.628  0.572       0.791
                Re-ranked (cross-encoder)   0.488   0.605  0.571       0.791
without headers Baseline (bi-encoder)       0.372   0.605  0.499       0.767
                Re-ranked (cross-encoder)   0.453   0.593  0.545       0.767

**What the metrics show**
- With headers (the main setup), re-ranking is roughly neutral overall: hit @1 rises slightly (0.477 → 0.488), hit @3
  falls slightly (0.628 → 0.605), MRR is unchanged (0.572 → 0.571).
- Removing the headers makes the bi-encoder clearly worse (hit @1 0.477 → 0.372, MRR 0.572 → 0.499). Re-ranking then
  recovers a good part of that (hit @1 0.372 → 0.453, MRR 0.499 → 0.545).
- So the headers do much of the work a re-ranker would otherwise do. With them, the bi-encoder is already strong and the
  cross-encoder has less to fix; without them, the cross-encoder's benefit is easier to see.
- Recall @10 (about 0.79) is the ceiling for both pipelines: about one query in five never has the right chunk in the top 10.

## 7. Answer 1 vs Answer 2

For each failure query from section 6: which clauses each pipeline gave the LLM, whether a gold (correct) clause was
among them, and both answers side by side. Answers come from the cache, so they are the same ones shown in section 6.

The **label** column says what went wrong in the weaker answer. It is filled in by hand in `ANSWER_LABELS` below, using
one of: *missing clause* (the deciding rule wasn't in the context, so the answer refuses or leaves it out),
*misstated rule* (the answer says something the Laws don't), *hallucination* (facts not in the context),
*no real difference*.

In [42]:
COMPARE_QUERIES = [
    ("re-rank helps", "Can the bowler Mankad the non-striker?"),
    ("re-rank helps", "What happens if the bowler bowls a beamer?"),
    ("re-rank helps", "Can play stop because of bad light?"),
    ("re-rank helps", "Is a grubber that rolls along the ground a legal delivery?"),
    ("re-rank helps", "What is the penalty for a slow over rate?"),
    ("re-rank hurts", "When can byes not be scored?"),
    ("re-rank hurts", "Can a batter be given out for hitting the ball a second time on purpose?"),
]

# Filled in by hand after reading the answers (label options above); the weaker answer is named in brackets
ANSWER_LABELS = {
    "Can the bowler Mankad the non-striker?":
        "missing clause (Answer 1 refuses: 38.3 not in the baseline top 3)",
    "What happens if the bowler bowls a beamer?":
        "missing clause (Answer 1 refuses: 41.7 not in the baseline top 3)",
    "Can play stop because of bad light?":
        "missing clause, mild (Answer 1 is right but incomplete: no 2.7, umpires decide fitness for play)",
    "Is a grubber that rolls along the ground a legal delivery?":
        "missing clause (Answer 1 refuses: 21.7 not in the baseline top 3)",
    "What is the penalty for a slow over rate?":
        "missing clause (Answer 1 refuses: 41.9 not in the baseline top 3)",
    "When can byes not be scored?":
        "missing clause (Answer 2 refuses: re-ranking dropped 23.1)",
    "Can a batter be given out for hitting the ball a second time on purpose?":
        "missing clause (Answer 2 only has the exceptions; it says the main rule 34.1 is missing)",
}

gold_of = {q: g for _, q, g in SCREEN_QUERIES}
rows = []
for case, q in COMPARE_QUERIES:
    p1 = pipeline_baseline(q)
    p2 = pipeline_rerank(q, candidates=p1["candidates"])
    gold = gold_of[q]
    rows.append({
        "case": case,
        "query": q,
        "gold": ", ".join(gold),
        "P1 top 3": ", ".join(p1["context"]["clause"]),
        "P1 has gold": p1["context"]["clause"].isin(gold).any(),
        "P2 top 3": ", ".join(p2["context"]["clause"]),
        "P2 has gold": p2["context"]["clause"].isin(gold).any(),
        "Answer 1 (baseline)": p1["answer"],
        "Answer 2 (re-ranked)": p2["answer"],
        "label": ANSWER_LABELS[q],
    })
answers_df = pd.DataFrame(rows)
answers_df.to_csv("output/answer_comparison.csv", index=False, encoding="utf-8-sig")   # easier to read in Excel

(answers_df.style
    .set_properties(**{"white-space": "pre-wrap", "text-align": "left", "vertical-align": "top"})
    .set_properties(subset=["Answer 1 (baseline)", "Answer 2 (re-ranked)"], **{"min-width": "320px"})
    .hide(axis="index"))

case,query,gold,P1 top 3,P1 has gold,P2 top 3,P2 has gold,Answer 1 (baseline),Answer 2 (re-ranked),label
re-rank helps,Can the bowler Mankad the non-striker?,38.3,"30.3, 32.1, 21.4",False,"38.3, 30.3, 32.1",True,The context does not say.,"“Mankad” refers to running out the non-striker for leaving their ground early, which is covered by [1] Law 38.3. Yes, but only within a limited window. Under [1] 38.3.1, if the non-striker is out of their ground from the moment the ball comes into play until the bowler’s arm reaches the highest point of their normal bowling action, they are out Run out if the wicket is broken by the bowler throwing the ball at the wicket or by the bowler’s hand holding the ball, whether or not the ball is subsequently delivered: [1] “If the non-striker is out of their ground at any time from the moment the ball comes into play until the moment the bowler’s arm reaches the highest point of their normal bowling action in the delivery swing, the non-striker is liable to be Run out. In these circumstances, the non-striker will be out Run out if they are out of their ground when the wicket is broken by the bowler throwing the ball at the wicket or by the bowler’s hand holding the ball, whether or not the ball is subsequently delivered.” However, [1] 38.3.1.1 states that once the bowler has reached that point, it is no longer possible to run out the non-striker under this Law: [1] “once the bowler has reached that point it is no longer possible for the bowler to run out the non-striker under this Law.”",missing clause (Answer 1 refuses: 38.3 not in the baseline top 3)
re-rank helps,What happens if the bowler bowls a beamer?,41.7,"17.6, 41.13, 41.8",False,"41.8, 41.7, 41.6",True,The context does not say.,"A beamer is a non-landing delivery, so it matches passage [2] (Law 41.7, dangerous and unfair non-landing deliveries). The context only covers a deliberate beamer: if the umpire considers the bowler deliberately bowled a non-landing delivery deemed unfair, the caution and warning are dispensed with; the umpire immediately calls and signals No ball; when the ball is dead, directs the captain to suspend the bowler immediately from bowling and informs the other umpire; the suspended bowler shall not bowl again in the match; and the over is completed by another bowler who neither bowled part of the previous over nor may bowl part of the next over. [2] ""If the umpire considers that a bowler deliberately bowled a non-landing delivery... the caution and warning in 41.7.3 shall be dispensed with. The umpire shall immediately call and signal No ball. when the ball is dead, direct the captain of the fielding side to suspend the bowler immediately from bowling and inform the other umpire for the reason for this action. The suspended bowler shall not be allowed to bowl again in the match. If applicable, the over shall be completed by another bowler, who shall neither have bowled any part of the previous over, nor be allowed to bowl any part of the next over."" The context does not cover what happens for a non-deliberate beamer.",missing clause (Answer 1 refuses: 41.7 not in the baseline top 3)
re-rank helps,Can play stop because of bad light?,"2.7, 2.8","12.9, 11.6, 12.7",False,"11.6, 12.9, 2.7",True,"Yes. [1] ""the players leave the field for adverse conditions of ground, weather or light, or in exceptional circumstances, and no further play is possible."" [2] also allows a stoppage ""because of adverse conditions of ground, weather or light.""","Yes. The umpires together decide whether conditions of light make play dangerous or unreasonable [3] ""It is solely for the umpires together to decide whether either conditions of ground, weather or light or exceptional circumstances mean that it would be dangerous or unreasonable for play to take place."" A stoppage can occur because of adverse conditions of light [1] ""If, because of adverse conditions of ground, weather or light... a stoppage occurs..."" and if players leave the 

## 8. Latency profiling

Each stage is timed separately with `time.perf_counter()`:
- **query embedding** (bi-encoder, one short text)
- **vector search** (one matrix product over 432 chunks + sorting)
- **cross-encoding** (10 query–chunk pairs)
- **LLM generation** (one API call to OpenRouter)

Retrieval and re-ranking are cheap to repeat, so they are timed over 10 queries × 5 repeats after a warm-up run
(the first call of a model is slower). LLM calls are slow and rate-limited on the free tier, so the end-to-end runs use
5 queries × 2 pipelines, without the answer cache, with a short pause between calls (the pause is not timed).
Everything runs on CPU.

In [43]:
LATENCY_QUERIES = SANITY_QUERIES + [
    "Can the bowler Mankad the non-striker?",
    "What happens if the bowler bowls a beamer?",
    "Can play stop because of bad light?",
    "Is a grubber that rolls along the ground a legal delivery?",
    "What is the penalty for a slow over rate?",
    "When can byes not be scored?",
    "Can a batter be given out for hitting the ball a second time on purpose?",
]
N_REPEATS = 5

# Warm-up: the first call of each model is slower (memory allocation, caches)
_c, _ = retrieve_top10(LATENCY_QUERIES[0])
rerank(LATENCY_QUERIES[0], _c)

stage_rows = []
for _ in range(N_REPEATS):
    for q in LATENCY_QUERIES:
        candidates, t_ret = retrieve_top10(q)
        _, t_ce = rerank(q, candidates)
        stage_rows.append({**t_ret, **t_ce})
stage_ms = pd.DataFrame(stage_rows) * 1000
print(f"{len(stage_ms)} timed runs of retrieval + re-ranking")
stage_ms.describe().loc[["mean", "50%", "min", "max"]].round(1)

50 timed runs of retrieval + re-ranking


,embed_query,vector_search,cross_encode
mean,26.6,0.2,310.7
50%,27.9,0.2,313.4
min,18.3,0.1,269.5
max,34.3,0.2,349.8


In [44]:
LLM_TIMING_QUERIES = LATENCY_QUERIES[:5]
PAUSE_S = 4   # pause between API calls for the free tier's rate limit (not part of the timings)

generate(LLM_TIMING_QUERIES[0], retrieve_top10(LLM_TIMING_QUERIES[0])[0].head(TOP_K_CONTEXT), use_cache=False)  # warm-up

e2e_rows = []
for q in LLM_TIMING_QUERIES:
    for name, run in [("Pipeline 1 (baseline)", pipeline_baseline), ("Pipeline 2 (re-rank)", pipeline_rerank)]:
        time.sleep(PAUSE_S)
        t0 = time.perf_counter()
        result = run(q, use_cache=False)          # real API call, retrieval included
        total = time.perf_counter() - t0
        e2e_rows.append({"pipeline": name, **result["timings"], "total": total})
e2e_ms = pd.DataFrame(e2e_rows).set_index("pipeline") * 1000
e2e_ms.round(1)

,embed_query,vector_search,llm,total,cross_encode
pipeline,,,,,
Pipeline 1 (baseline),173.3,0.2,128680.5,128857.0,NaN
Pipeline 2 (re-rank),114.2,0.1,2257.9,2642.9,266.6
Pipeline 1 (baseline),185.5,0.1,11813.3,12001.2,NaN
Pipeline 2 (re-rank),190.5,0.1,4069.6,4557.0,292.3
Pipeline 1 (baseline),153.5,0.1,4807.7,4963.7,NaN
Pipeline 2 (re-rank),136.1,0.2,2933.7,3384.4,309.1
Pipeline 1 (baseline),138.8,0.1,29036.7,29177.8,NaN
Pipeline 2 (re-rank),157.1,0.2,11180.2,11654.6,313.6
Pipeline 1 (baseline),112.3,0.1,5698.2,5812.8,NaN


In [45]:
def stats(series):
    return {"mean ms": series.mean(), "median ms": series.median(), "min ms": series.min(), "max ms": series.max()}


retrieval_only = stage_ms["embed_query"] + stage_ms["vector_search"]
latency_table = pd.DataFrame({
    "Query embedding (bi-encoder)": stats(stage_ms["embed_query"]),
    "Vector search (432 chunks)": stats(stage_ms["vector_search"]),
    "Cross-encoding (10 pairs)": stats(stage_ms["cross_encode"]),
    "LLM generation": stats(e2e_ms["llm"]),
    "Retrieval only (Pipeline 1 without LLM)": stats(retrieval_only),
    "Retrieval + re-ranking (Pipeline 2 without LLM)": stats(retrieval_only + stage_ms["cross_encode"]),
    "End to end, Pipeline 1 (baseline)": stats(e2e_ms.loc["Pipeline 1 (baseline)", "total"]),
    "End to end, Pipeline 2 (re-rank)": stats(e2e_ms.loc["Pipeline 2 (re-rank)", "total"]),
}).T.round(1)

# The free tier sometimes queues a request for tens of seconds, which has nothing to do with the pipeline.
# LLM time doesn't depend on which pipeline chose the chunks, so a typical end-to-end time is estimated from medians:
llm_median = e2e_ms["llm"].median()
typical = pd.DataFrame({
    "typical end to end (median-based) ms": {
        "Pipeline 1 (baseline)": retrieval_only.median() + llm_median,
        "Pipeline 2 (re-rank)": (retrieval_only + stage_ms["cross_encode"]).median() + llm_median,
    }}).round(1)

overhead = stage_ms["cross_encode"].mean()
print(f"Re-ranking adds {overhead:.0f} ms per query: {overhead / retrieval_only.mean():.1f}x the retrieval time, "
      f"but only {100 * overhead / (retrieval_only.median() + llm_median):.1f}% of a typical end-to-end time.")
print(f"One-off index build (not per query): {index_time:.1f} s for {len(chunks_df)} chunks")
display(latency_table)
typical

Re-ranking adds 311 ms per query: 11.6x the retrieval time, but only 3.7% of a typical end-to-end time.
One-off index build (not per query): 26.4 s for 432 chunks


,mean ms,median ms,min ms,max ms
Query embedding (bi-encoder),26.6,27.9,18.3,34.3
Vector search (432 chunks),0.2,0.2,0.1,0.2
Cross-encoding (10 pairs),310.7,313.4,269.5,349.8
LLM generation,21450.9,8439.2,2257.9,128680.5
Retrieval only (Pipeline 1 without LLM),26.8,28.1,18.4,34.5
Retrieval + re-ranking (Pipeline 2 without LLM),337.5,341.2,296.8,380.9
"End to end, Pipeline 1 (baseline)",36162.5,12001.2,4963.7,128857.0
"End to end, Pipeline 2 (re-rank)",7335.2,4557.0,2642.9,14437.2


,typical end to end (median-based) ms
Pipeline 1 (baseline),8467.3
Pipeline 2 (re-rank),8780.4


**Notes on the latency results**
- Retrieval is very fast: embedding the query takes tens of milliseconds and searching 432 vectors well under 1 ms,
  because the chunk vectors were computed in advance (the one-off index build takes ~25 s).
- Re-ranking 10 pairs adds a few hundred milliseconds, roughly 10× the retrieval time, because the cross-encoder runs a
  full model pass for every (query, chunk) pair at query time.
- The LLM dominates end-to-end time (seconds, not milliseconds), so re-ranking is only a small share of the total.
- LLM times on the free tier vary a lot between calls (compare the min and max columns): a request can be queued or
  rate-limited for tens of seconds. That is why medians are used for the typical end-to-end estimate.
- Exact numbers change from run to run; the proportions stay the same.

## 9. Summary notes

**Setup**
- Document: MCC Laws of Cricket (2017 Code, 4th Edition, 2026), PDF pages 5–82 → 432 chunks, one per clause
  (max 200 tokens), each starting with its Law and clause title.
- Pipeline 1: `bge-small-en-v1.5` top 10 → first 3 → LLM. Pipeline 2: same top 10 → `ms-marco-MiniLM-L-6-v2` → top 3 → LLM.
- LLM: `nvidia/nemotron-3.5-super-vl:free` via OpenRouter, same prompt for both pipelines, answers only from the
  passages with quotes (leakage tests in 5.1 passed).

**Where the results are**
- Rank shifts for each failure query: section 6 (rank tables) and 7 (comparison table).
- How often re-ranking helps or hurts over all 86 screened queries: 6.1 and 6.10.
- Why the effect is small on this document (chunk headers): 6.11. Wider pool and larger cross-encoder: 6.9.
- Latency per stage and end to end: section 8.

**Limitations**
- The gold clauses for the screening queries were chosen by hand.
- One document and 86 queries; the failure examples were selected from the screening, so they show how re-ranking can
  help or hurt, while 6.10 shows how often.
- Free-tier LLM: response times vary and answers are not perfectly deterministic (answers are cached in `output/`).